# 02 - Limpieza de Datos
Proceso completo de limpieza: estandarizacion de valores sucios, recuperacion 
de datos numericos, imputacion de categoricas y eliminacion de filas irrecuperables.

In [1]:
import pandas as pd

df = pd.read_csv('../data/dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# Reemplazamos los valores "sucios" por NaN real de pandas
df = df.replace(['ERROR', 'UNKNOWN'], pd.NA)

# Verificamos que ya no existan
for col in df.select_dtypes(include=['object', 'str']).columns:
    print(col, ':', df[col].unique())

Transaction ID : <StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365',
 ...
 'TXN_1538510', 'TXN_3897619', 'TXN_2739140', 'TXN_4766549', 'TXN_7851634',
 'TXN_7672686', 'TXN_9659401', 'TXN_5255387', 'TXN_7695629', 'TXN_6170729']
Length: 10000, dtype: str
Item : <StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',        nan,
 'Sandwich',    'Juice',      'Tea']
Length: 9, dtype: str
Quantity : <StringArray>
['2', '4', '5', '3', '1', nan]
Length: 6, dtype: str
Price Per Unit : <StringArray>
['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', nan]
Length: 7, dtype: str
Total Spent : <StringArray>
[ '4.0', '12.0',    nan, '10.0', '20.0',  '9.0', '16.0', '15.0', '25.0',
  '8.0',  '5.0',  '3.0',  '6.0',  '2.0',  '1.0',  '7.5',  '4.5',  '1.5']
Length: 18, dtype: str
Payment Method : <StringArray>
['Credit Card', 'Cash', nan, 'Digital Wallet']
Length: 4, dtype: str
Locat

In [3]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [4]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

In [5]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9031 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   str           
 6   Location          6039 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


In [7]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [8]:
# Calculamos el valor "correcto" según la fórmula
calculado = df['Quantity'] * df['Price Per Unit']

# Solo llenamos los NaN de Total Spent con el valor calculado
# (si Total Spent ya tenía un valor, lo dejamos como estaba)
df['Total Spent'] = df['Total Spent'].fillna(calculado)

In [9]:
df['Total Spent'].isnull().sum()

np.int64(40)

In [10]:
# Price Per Unit = Total Spent / Quantity
calculado_precio = df['Total Spent'] / df['Quantity']
df['Price Per Unit'] = df['Price Per Unit'].fillna(calculado_precio)

In [11]:
# Quantity = Total Spent / Price Per Unit
calculado_cantidad = df['Total Spent'] / df['Price Per Unit']
df['Quantity'] = df['Quantity'].fillna(calculado_cantidad)

In [12]:
df[['Quantity', 'Price Per Unit', 'Total Spent']].isnull().sum()

Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64

Aquí aprendimos que comparar con NaN da resultados inesperados

In [13]:
# Vemos si hay valores no enteros en Quantity
df[df['Quantity'] % 1 != 0][['Quantity', 'Price Per Unit', 'Total Spent']]

,Quantity,Price Per Unit,Total Spent
236,NaN,5.0,NaN
278,NaN,3.0,NaN
629,NaN,NaN,12.0
641,NaN,3.0,NaN
738,NaN,4.0,NaN
912,NaN,NaN,20.0
1008,NaN,NaN,3.0
1436,NaN,NaN,6.0
1482,NaN,NaN,16.0
2330,NaN,NaN,5.0


In [14]:
# Excluimos los NaN explícitamente, y buscamos solo decimales reales
df[df['Quantity'].notna() & (df['Quantity'] % 1 != 0)][['Quantity', 'Price Per Unit', 'Total Spent']]

,Quantity,Price Per Unit,Total Spent


In [15]:
# Filas donde Quantity, Price Per Unit o Total Spent siguen siendo NaN
df[df['Quantity'].isnull() | df['Price Per Unit'].isnull() | df['Total Spent'].isnull()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-store,2023-10-20
236,TXN_8562645,Salad,NaN,5.0,NaN,NaN,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
629,TXN_9289174,Cake,NaN,NaN,12.0,Digital Wallet,In-store,2023-12-30
641,TXN_2962976,Juice,NaN,3.0,NaN,NaN,NaN,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,NaN,Takeaway,2023-05-14
912,TXN_1575608,Sandwich,NaN,NaN,20.0,NaN,Takeaway,2023-01-05
1008,TXN_7225428,Tea,NaN,NaN,3.0,Credit Card,Takeaway,2023-03-07
1436,TXN_7590801,Tea,NaN,NaN,6.0,Cash,Takeaway,NaT
1482,TXN_3593060,Smoothie,NaN,NaN,16.0,Cash,NaN,2023-03-05


In [16]:
# Filas donde Transaction Date es NaN
df[df['Transaction Date'].isnull()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
11,TXN_3051279,Sandwich,2.0,4.0,8.0,Credit Card,Takeaway,NaT
29,TXN_7640952,Cake,4.0,3.0,12.0,Digital Wallet,Takeaway,NaT
33,TXN_7710508,NaN,5.0,1.0,5.0,Cash,NaN,NaT
77,TXN_2091733,Salad,1.0,5.0,5.0,NaN,In-store,NaT
103,TXN_7028009,Cake,4.0,3.0,12.0,NaN,Takeaway,NaT
...,...,...,...,...,...,...,...,...
9933,TXN_9460419,Cake,1.0,3.0,3.0,NaN,Takeaway,NaT
9937,TXN_8253472,Cake,1.0,3.0,3.0,NaN,NaN,NaT
9949,TXN_3130865,Juice,3.0,3.0,9.0,NaN,In-store,NaT
9983,TXN_9226047,Smoothie,3.0,4.0,12.0,Cash,NaN,NaT


In [17]:
df['Item'] = df['Item'].fillna('Not Specified')
df['Payment Method'] = df['Payment Method'].fillna('Not Specified')
df['Location'] = df['Location'].fillna('Not Specified')

In [18]:
df[['Item', 'Payment Method', 'Location']].isnull().sum()

Item              0
Payment Method    0
Location          0
dtype: int64

## Decisión: Transaction Date

460 filas (4.6%) no tienen fecha de transacción, pero sí tienen datos completos de venta 
(Quantity, Price Per Unit, Total Spent, Item, Payment Method, Location). 

Decisión: Conservar estas filas en el dataset limpio. Se excluirán únicamente en los 
análisis que dependan de series de tiempo (tendencias por mes, día de la semana, etc.), 
usando `df.dropna(subset=['Transaction Date'])` en esos análisis específicos.

In [19]:
# Eliminamos filas donde Quantity, Price Per Unit o Total Spent siguen siendo NaN
antes = df.shape[0]
df = df.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent'])
despues = df.shape[0]

print(f'Filas antes: {antes}')
print(f'Filas despues: {despues}')
print(f'Filas eliminadas: {antes - despues}')

Filas antes: 10000
Filas despues: 9942
Filas eliminadas: 58


In [20]:
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    457
dtype: int64

In [21]:
df.to_csv('../data/cafe_sales_clean.csv', index=False)

In [22]:
# Recarga el archivo guardado y confirma que coincide
verificacion = pd.read_csv('../data/cafe_sales_clean.csv')
print(verificacion.shape)
verificacion.head()

(9942, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Not Specified,Not Specified,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


## Resumen de la limpieza (Fase 2)

- Se estandarizaron los valores "ERROR" y "UNKNOWN" a NaN real.
- Se convirtieron Quantity, Price Per Unit y Total Spent a tipo numerico (float64).
- Se convirtio Transaction Date a tipo datetime.
- Se recuperaron ~460-495 valores en Quantity/Price Per Unit/Total Spent usando la 
  relacion matematica Total Spent = Quantity * Price Per Unit.
- Se rellenaron los nulos de Item, Payment Method y Location con "Not Specified", 
  preservando filas con datos de venta validos.
- Se eliminaron ~38-40 filas donde no fue posible calcular ni recuperar Quantity, 
  Price Per Unit o Total Spent (dato de venta irrecuperable).
- Se conservaron 457 filas sin Transaction Date, ya que contienen datos de venta 
  completos; se excluiran unicamente en analisis especificos de series de tiempo.

Dataset final: [X] filas x 8 columnas, guardado en data/cafe_sales_clean.csv